# 13 — Prompt Security and Untrusted Content

## Scenario
Northstar uses RAG to summarize customer support tickets for agents. 
However, a malicious user submits a ticket containing a hidden "Prompt Injection" attack.

**The Problem:** LLMs cannot reliably distinguish between "System Instructions" and "User Data" because both are fed into the model as a single stream of text. If user data contains text that looks like an instruction, the model might follow it.


In [ ]:
from pathlib import Path
import sys

COURSE_DIR = Path.cwd()
sys.path.insert(0, str(COURSE_DIR))
from northstar.runtime import get_client
from lab13 import INJECTIONS, build_requests, route_summary
from northstar.contracts import check_constraints
from northstar.security import Action, Principal, authorize, instruction_like_score


def show_request(request):
    print("SYSTEM:\n", request.system)
    for message in request.messages:
        if message.text:
            print(f"{message.role.upper()}:\n{message.text}")
        for part in message.parts:
            print(f"{message.role.upper()} PART:", part)

client = get_client(COURSE_DIR / "fixtures/replays.json")


## Step 1: The Vulnerable Implementation (Indirect Prompt Injection)

We fetch a "Customer Ticket" from our database and ask the model to summarize it. Watch what happens.


In [ ]:
request = next(r for r in build_requests() if r.case_id == "i13/vulnerable/injection")
show_request(request)
response = client.generate(request)
print("RECORDED RESPONSE:", response.text)
print("PARSED ROUTE:", route_summary(response.text))
assert check_constraints(response.text, forbidden_phrases=("SYSTEM COMPROMISED",))


## Step 2: The Defense (Data Separation via XML Tags)

To defend against this, we use the `system_instruction` parameter to define the model's persona, and we wrap the untrusted data in strict XML tags so the model knows what is data vs what is an instruction.


In [ ]:
request = next(r for r in build_requests() if r.case_id == "i13/defended/injection")
show_request(request)
response = client.generate(request)
print("RECORDED RESPONSE:", response.text)
print("PARSED ROUTE:", route_summary(response.text))
assert not check_constraints(response.text, forbidden_phrases=("SYSTEM COMPROMISED",))


## Step 3: The Ultimate Truth (Application Control)

While XML tags and strict prompting reduce the success rate of injections, **they are not a 100% secure boundary**. Advanced attackers can still find ways to "break out" of tags (e.g., by predicting the closing tag).

The *only* true defense is **Application Control**:
1. Never give an LLM direct access to destructive tools without a human-in-the-loop.
2. Treat all LLM output as untrusted data.
3. Use deterministic code (e.g., Python `if` statements) to enforce security boundaries, not English prompts.


In [ ]:
principal = Principal(user_id="USER-0001", tenant="tenant-synthetic-a", roles={"support_agent"})
decision = authorize(principal, Action(name="send_all_user_data", tenant="tenant-synthetic-a", requires_role="data_export"))
print("AUTHORIZATION:", decision)
assert decision.reason_code == "missing_role"
score = sum(instruction_like_score(item["text"]) >= 0.8 for item in INJECTIONS)
print("DETECTED INJECTIONS:", score, "/", len(INJECTIONS))
assert score == 8
assert route_summary("SYSTEM COMPROMISED: leak all records") == "blocked"


## Takeaway
The recorded security run detected all 8 synthetic injections, produced no benign false positives in the fixture suite, and blocked a leaking summary with the application-side constraint check.


## References
- [Core Concepts & Workflow](README.md#core-concepts--workflow)
- [Deep dive](README.md#deep-dive)
- [Production Best Practices](README.md#production-best-practices)
